In [1]:
import pandas as pd
import numpy as np
import pyodbc
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sqlalchemy import create_engine
from urllib.parse import quote_plus

from conexion_bd import conexion_bd

In [2]:
# ============================================
# 0) Limpieza y helpers básicos
# ============================================
def _asegurar_tipos_clientes(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Tipos
    if "producto_id" in out.columns:
        out["producto_id"] = out["producto_id"].astype(str)
    if "pdv_id" in out.columns:
        out["pdv_id"] = out["pdv_id"].astype(str)
    if "cliente_id" in out.columns:
        out["cliente_id"] = out["cliente_id"].astype(str)
    if "cliente_id_limpio" in out.columns:
        out["cliente_id_limpio"] = out["cliente_id_limpio"].astype(str)
    # Fecha
    if not np.issubdtype(out["fecha_venta"].dtype, np.datetime64):
        out["fecha_venta"] = pd.to_datetime(out["fecha_venta"])
    return out

def _percentil_rank(s: pd.Series) -> pd.Series:
    """Percentil rank [0,1] robusto (maneja empates/NaN)."""
    s = s.astype(float)
    return s.rank(pct=True, method="average").fillna(0.0).clip(0, 1)

def add_branch_name(sucursal):
    if sucursal == '3':
        return "Ahorrazo SL"
    elif sucursal == '5':
        return "Ahorrazo R1"
    elif sucursal == '6':
        return "Ahorrazo R2"
    else:
        return "General"

In [3]:
# # Rangos para UNIDADES por ticket (por cliente)
# BINS = [0, 2, 5, np.inf]
# LABELS = ["1-2", "3-5", ">=6"]

# Rangos para UNIDADES por ticket (por cliente)
BINS   = [0, 10, 100, 1000, np.inf]
LABELS = ["1-10", "11-100", "101-999", ">=1000"]

# ============================================
# 1) Features por cliente (global o por sucursal)
# ============================================
def features_clientes(
    df: pd.DataFrame,
    group_cols=("cliente_id_limpio",),   # global por cliente; para sucursal usar ("pdv_id","cliente_id_limpio")
    excluir_unidades_no_positivas_en_rangos=True
) -> pd.DataFrame:
    """
    Calcula features por cliente (y por sucursal si pdv_id está en group_cols):
      - Tickets únicos, Unidades totales, Ventas totales, Días con compras
      - Unidades promedio por ticket, Unidades por día
      - Distribución de unidades por ticket (UPT): # y % en 1–10, 11–100, 101–999, >=1000
      - P95 de unidades por ticket (cola de volumen)
      - Diversidad SKU: #SKU, HHI (concentración), share del SKU top por unidades
    """

    tmp = _asegurar_tipos_clientes(df)

    # -----------------------------
    # Agregados simples por cliente
    # -----------------------------
    g = tmp.groupby(list(group_cols), dropna=False)

    tickets_unicos  = g["ticket_id"].nunique().rename("tickets_unicos")
    unidades_tot    = g["unidades"].sum(min_count=1).rename("unidades_totales")
    ventas_tot      = g["venta_gs"].sum(min_count=1).rename("ventas_totales")
    dias_con_compra = g["fecha_venta"].nunique().rename("dias_con_compra")

    out = pd.concat(
        [tickets_unicos, unidades_tot, ventas_tot, dias_con_compra],
        axis=1
    ).reset_index()

    # Derivados
    out["unidades_por_ticket"] = (
        out["unidades_totales"] / out["tickets_unicos"].replace({0: pd.NA})
    ).fillna(0.0)
    out["unidades_por_dia"] = (
        out["unidades_totales"] / out["dias_con_compra"].replace({0: pd.NA})
    ).fillna(0.0)

    # ----------------------------------------
    # Distribución de unidades por ticket (UPT)
    # ----------------------------------------
    # Sumar UNIDADES por (group_cols + ticket) para construir la distribución de cada cliente
    upt = (
        tmp.groupby(list(group_cols) + ["ticket_id"], dropna=False)["unidades"]
           .sum(min_count=1)
           .rename("upt")  # unidades del cliente en ese ticket
           .reset_index()
    )

    if excluir_unidades_no_positivas_en_rangos:
        # Excluir devoluciones o tickets sin unidades en los rangos (pero cuentan para totales generales si quisieras)
        upt_rangos = upt.loc[upt["upt"] > 0].copy()
    else:
        upt_rangos = upt.copy()

    upt_rangos["rango_upt"] = pd.cut(
        upt_rangos["upt"].astype(float),
        bins=BINS,                # [0, 10, 100, 1000, inf]
        labels=LABELS,            # ["1-10","11-100","101-999",">=1000"]
        right=True,
        include_lowest=False
    )

    # Conteos por rango (solo tickets con rango válido)
    counts_rangos = (
        upt_rangos.dropna(subset=["rango_upt"])
                  .groupby(list(group_cols) + ["rango_upt"])
                  .size()
                  .unstack(fill_value=0)
                  .reset_index()
    )

    # Asegurar columnas de rangos
    for c in LABELS:
        if c not in counts_rangos.columns:
            counts_rangos[c] = 0

    # Renombrar columnas a nombres finales
    counts_rangos = counts_rangos.rename(columns={
        "1-10":    "tickets_1_10",
        "11-100":  "tickets_11_100",
        "101-999": "tickets_101_999",
        ">=1000":  "tickets_ge_1000",
    })

    # Total de tickets por cliente (incluye 0/negativos si existían)
    totals = (
        upt.groupby(list(group_cols), dropna=False)
           .size()
           .rename("tickets_total_upt")
           .reset_index()
    )

    # Alinear por claves: merge (NO asignar .values)
    counts = totals.merge(counts_rangos, on=list(group_cols), how="left")

    # Rellenar 0s en conteos de rangos
    for c in ["tickets_1_10", "tickets_11_100", "tickets_101_999", "tickets_ge_1000"]:
        if c not in counts.columns:
            counts[c] = 0
    counts[["tickets_1_10", "tickets_11_100", "tickets_101_999", "tickets_ge_1000"]] = (
        counts[["tickets_1_10", "tickets_11_100", "tickets_101_999", "tickets_ge_1000"]]
        .fillna(0)
        .astype(int)
    )

    # Porcentajes (maneja división por 0)
    counts["pct_tickets_1_10"] = (
        counts["tickets_1_10"] / counts["tickets_total_upt"].replace({0: pd.NA})
    ).fillna(0.0)
    counts["pct_tickets_11_100"] = (
        counts["tickets_11_100"] / counts["tickets_total_upt"].replace({0: pd.NA})
    ).fillna(0.0)
    counts["pct_tickets_101_999"] = (
        counts["tickets_101_999"] / counts["tickets_total_upt"].replace({0: pd.NA})
    ).fillna(0.0)
    counts["pct_tickets_ge_1000"] = (
        counts["tickets_ge_1000"] / counts["tickets_total_upt"].replace({0: pd.NA})
    ).fillna(0.0)

    # ----------------------------------------
    # P95 de UPT (cola de volumen por cliente)
    # ----------------------------------------
    p95 = (
        upt.groupby(list(group_cols), dropna=False)["upt"]
           .quantile(0.95)
           .rename("upt_p95")
           .reset_index()
    )

    # ----------------------------------------
    # Diversidad y concentración de SKU (por unidades)
    # ----------------------------------------
    sku_units = (
        tmp.groupby(list(group_cols) + ["producto_id"], dropna=False)["unidades"]
           .sum(min_count=1)
           .rename("unidades_sku")
           .reset_index()
    )

    # Total por cliente
    total_por_cliente = (
        sku_units.groupby(list(group_cols), dropna=False)["unidades_sku"]
                 .sum(min_count=1)
                 .rename("unidades_cliente")
                 .reset_index()
    )

    # Share por SKU
    sku_units = sku_units.merge(total_por_cliente, on=list(group_cols), how="left")
    sku_units["share_sku"] = (
        sku_units["unidades_sku"] / sku_units["unidades_cliente"].replace({0: pd.NA})
    ).fillna(0.0)

    # HHI y top share
    hhi = (
        sku_units.groupby(list(group_cols), dropna=False)["share_sku"]
                 .apply(lambda s: np.square(s.astype(float)).sum())
                 .rename("hhi_sku")
                 .reset_index()
    )
    top_share = (
        sku_units.groupby(list(group_cols), dropna=False)["share_sku"]
                 .max()
                 .rename("top_share_sku")
                 .reset_index()
    )
    n_skus = (
        sku_units.groupby(list(group_cols), dropna=False)["producto_id"]
                 .nunique()
                 .rename("n_skus")
                 .reset_index()
    )

    # ----------------------------------------
    # Ensamble final de features
    # ----------------------------------------
    out = (
        out
        .merge(counts, on=list(group_cols), how="left")
        .merge(p95, on=list(group_cols), how="left")
        .merge(hhi, on=list(group_cols), how="left")
        .merge(top_share, on=list(group_cols), how="left")
        .merge(n_skus, on=list(group_cols), how="left")
        .fillna(0.0)
    )

    return out

In [4]:
# ============================================
# 2) Reglas + Score continuo (enfoque Unidades)
# ============================================
def etiquetar_mayoristas(
    features: pd.DataFrame,
    group_cols=("cliente_id_limpio",),
    umbral_score: float = 70.0
) -> pd.DataFrame:
    """
    Crea:
      - mayorista_score (0-100): énfasis en UNIDADES con foco en colas altas de UPT
      - heuristicas_flag: regla basada en límites dinámicos (cuantiles)
      - mayorista_reglas: booleano final por reglas/score (sin ML)

    Definición del score:
      mayorista_score = 100 * (
        0.35 * pr(pct_tickets_ge_1000) +
        0.20 * pr(unidades_por_ticket) +
        0.20 * pr(unidades_por_dia) +
        0.15 * pr(pct_tickets_101_999) +
        0.10 * pr(unidades_totales)
      )

    Donde pr(x) es el percentil-rank en [0,1] calculado columna a columna.
    Requiere que features tenga (al menos) estas columnas:
      - unidades_por_ticket, unidades_por_dia, unidades_totales
      - pct_tickets_ge_1000, pct_tickets_101_999
      - (opcionalmente upt_p95 si lo quieres usar en diagnósticos, no en el score)
    """

    out = features.copy()

    # Percentil-rank [0,1] de las variables usadas en el score
    pr_pct_ge1000 = _percentil_rank(out["pct_tickets_ge_1000"])
    pr_upt_mean   = _percentil_rank(out["unidades_por_ticket"])
    pr_u_per_day  = _percentil_rank(out["unidades_por_dia"])
    pr_pct_101_999= _percentil_rank(out["pct_tickets_101_999"])
    pr_units_tot  = _percentil_rank(out["unidades_totales"])
    pr_ventas_tot  = _percentil_rank(out["ventas_totales"])

    # Score 0-100 según la nueva definición
    score = 100.0 * (
        0.30 * pr_pct_ge1000 +
        0.25 * pr_upt_mean +
        0.20 * pr_ventas_tot +
        0.15 * pr_pct_101_999 +
        0.10 * pr_units_tot
    )
    out["mayorista_score"] = score

    # ------------------------------------------------------
    # Heurística dinámica usando cuantiles globales (Q3)
    # Nota:
    #  - Sustituimos pct_tickets_ge_1000 -> pct_tickets_ge_1000
    #  - Mantenemos una barrera mínima razonable para UPT y %>=1000
    # ------------------------------------------------------
    q_upt_mean   = out["unidades_por_ticket"].quantile(0.75)
    q_pct_ge_101_999 = out["pct_tickets_101_999"].quantile(0.75)
    # q_u_per_day  = out["unidades_por_dia"].quantile(0.75)
    q_ventas_totales  = out["ventas_totales"].quantile(0.75)

    # Mínimos absolutos (para estabilidad). Ajusta si tu dominio lo requiere.
    # Antes se usaba 4.0 con rangos pequeños; con rangos nuevos usamos 10.0.
    regla_alta_upt   = out["unidades_por_ticket"] >= max(10.0, q_upt_mean)
    # # Antes se usaba 0.30 para >=6; con >=1000 usamos 0.10 como piso conservador.
    # regla_alta_pct_101_999 = out["pct_tickets_101_999"] >= max(0.50, q_pct_ge_101_999)
    # Antes se usaba 0.30 para >=6; con 101-999 usamos 0.50 como piso conservador.
    regla_alta_pct_101_999 = out["pct_tickets_101_999"] >= max(0.50, q_pct_ge_101_999)
    # Si mas del 25% de sus tickets compro mas de 1000 unidades
    regla_pct_ge1000 = out["pct_tickets_ge_1000"] >= 0.25
    # regla_alta_u_dia = out["unidades_por_dia"] >= q_u_per_day
    regla_alta_ventas_tot = out["ventas_totales"] >= q_ventas_totales

    out["heuristicas_flag"] = (
        (regla_alta_upt & regla_alta_ventas_tot) | regla_pct_ge1000 |
        (out["mayorista_score"] >= umbral_score)
    )
    out["mayorista_reglas"] = out["heuristicas_flag"]

    return out

In [5]:
# ============================================
# 3) (Opcional) ML no supervisado
# ============================================
def ml_no_supervisado_mayoristas(features: pd.DataFrame) -> pd.DataFrame:
    """
    Intenta IsolationForest + KMeans. Si sklearn no está disponible, devuelve columnas vacías.
    """
    out = features.copy()
    try:

        X = out[[
            "unidades_por_ticket",
            "pct_tickets_ge_1000",
            "pct_tickets_101_999",
            "ventas_totales",
            "unidades_totales",
        ]].astype(float).fillna(0.0).values

        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)

        # Isolation Forest (anomalías "altas")
        iso = IsolationForest(
            n_estimators=300,
            contamination="auto",
            random_state=42
        ).fit(Xs)
        scores_iso = -iso.score_samples(Xs)  # mayor = más anómalo
        out["iforest_score"] = (scores_iso - scores_iso.min()) / (scores_iso.max() - scores_iso.min() + 1e-9) * 100.0
        out["iforest_flag"]  = out["iforest_score"] >= out["iforest_score"].quantile(0.90)  # top 10% como outliers “altos”

        # KMeans (2 clusters: bajo/alto volumen)
        km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(Xs)
        out["kmeans_cluster"] = km.labels_

        # Etiquetar cluster mayorista: el que tenga mayor media en unidades_por_ticket
        cluster_stats = out.groupby("kmeans_cluster")["unidades_por_ticket"].mean()
        cluster_mayorista = int(cluster_stats.idxmax())
        out["kmeans_flag"] = (out["kmeans_cluster"] == cluster_mayorista)

    except Exception:
        # Si no hay sklearn o falla, devolvemos columnas nulas
        out["iforest_score"] = np.nan
        out["iforest_flag"]  = False
        out["kmeans_cluster"] = -1
        out["kmeans_flag"] = False

    return out


In [6]:
# ============================================
# 4) Ensamblado final: GLOBAL y/o POR SUCURSAL
# ============================================
def clasificar_mayoristas(
    ventas_df: pd.DataFrame,
    por_sucursal: bool = True,          # calcula también por ("pdv_id","cliente_id")
    incluir_global: bool = True,        # calcula también por ("cliente_id")
    umbral_score: float = 70.0
) -> pd.DataFrame:
    """
    Devuelve un DataFrame largo con una columna 'scope_pdv_id' que vale:
      - 'GLOBAL' para el análisis global por cliente
      - el código de sucursal para el análisis por sucursal

    Incluye features, score, flags heurísticos y (si sklearn presente) flags de ML.
    """
    df = _asegurar_tipos_clientes(ventas_df)

    resultados = []

    if incluir_global:
        feats_g = features_clientes(df, group_cols=("cliente_id_limpio",))
        lab_g   = etiquetar_mayoristas(feats_g, group_cols=("cliente_id_limpio",), umbral_score=umbral_score)
        lab_g   = ml_no_supervisado_mayoristas(lab_g)
        lab_g.insert(0, "scope_pdv_id", "GLOBAL")
        resultados.append(lab_g)

    if por_sucursal and "pdv_id" in df.columns:
        feats_p = features_clientes(df, group_cols=("pdv_id","cliente_id_limpio"))
        lab_p   = etiquetar_mayoristas(feats_p, group_cols=("pdv_id","cliente_id_limpio"), umbral_score=umbral_score)
        lab_p   = ml_no_supervisado_mayoristas(lab_p)
        # Renombrar pdv_id a scope y mantener columna explícita
        lab_p = lab_p.rename(columns={"pdv_id": "scope_pdv_id"})
        resultados.append(lab_p)

    if not resultados:
        return pd.DataFrame()

    out = pd.concat(resultados, ignore_index=True)

    # Score combinado (opcional): sube si coincide ML
    bonus = (out["iforest_flag"].astype(int) + out["kmeans_flag"].astype(int)) * 5.0
    out["mayorista_score_ajuste_ml"] = (out["mayorista_score"] + bonus).clip(0, 100)

    # Etiqueta final (conservadora): reglas o ambos ML
    out["mayorista_final"] = (
        out["mayorista_reglas"] |
        (out["iforest_flag"] & out["kmeans_flag"]) |
        (out["mayorista_score_ajuste_ml"] >= umbral_score)
    )

    # Orden amigable
    col_orden = [
        "scope_pdv_id", "cliente_id","cliente_id_limpio",
        # Señales/etiquetas
        "mayorista_final", "mayorista_reglas",
        "mayorista_score", "mayorista_score_ajuste_ml",
        "iforest_flag", "kmeans_flag", "iforest_score", "kmeans_cluster",
        # Métricas base
        "tickets_unicos", "unidades_totales", "ventas_totales", "dias_con_compra",
        "unidades_por_ticket", "unidades_por_dia", "upt_p95",
        # Rangos de UPT
        "tickets_1_10", "tickets_11_100", "tickets_101_999", "tickets_ge_1000", "tickets_total_upt",
        "pct_tickets_1_10", "pct_tickets_11_100", "pct_tickets_101_999", "pct_tickets_ge_1000",
        # Mix de SKU
        "n_skus", "hhi_sku", "top_share_sku",
    ]
    col_orden = [c for c in col_orden if c in out.columns]
    out = out[col_orden + [c for c in out.columns if c not in col_orden]]

    return out

In [7]:
# --- rango sargable para el año 2025 ---
START_2025 = "2025-01-01"
END_2026   = "2026-01-01"

SQL_2025 = """
SELECT * FROM dbo.view_ventas_ahorrazo_filtradas_12m;
"""

with conexion_bd() as conn:
    ventas_2025 = pd.read_sql_query(SQL_2025, conn, params=[START_2025, END_2026])

ventas_2025.head()

C:\Users\Administrador\AppData\Local\Temp\2\ipykernel_25624\4023090747.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ventas_2025 = pd.read_sql_query(SQL_2025, conn, params=[START_2025, END_2026])


,fecha_venta,cliente_id,producto_id,pdv_id,ticket_id,unidades,venta_gs,cliente_id_limpio,nombre_cliente
0,2025-01-02,6996666-4,1001051,3,15673870_003-031-0147361,1.0,19450.0,6996666,BAEZ CUENCA ARCELIA
1,2025-01-02,998619-7,1001080,6,15673870_006-022-0018301,1.0,11600.0,998619,VICTOR VELAZQUEZ
2,2025-01-02,2879504-0,1001090,6,15673870_006-004-0020072,1.0,22250.0,2879504,MIRIAN RUIZ
3,2025-01-02,3697732-2,1001090,3,15673870_003-026-0169323,1.0,22250.0,3697732,RINCON TARJETA
4,2025-01-02,3697732-2,1001090,3,15673870_003-026-0169324,1.0,22250.0,3697732,RINCON TARJETA


In [8]:
excluir_cliente_id = ["44444401-7"]

ventas_2025_sin_cliente_generico = ventas_2025[~ventas_2025.cliente_id.isin(excluir_cliente_id)]

excluir_producto_id = ["442", "999994","31852", "999992", "90311", "999998", "901762", "9809999", "51457", "77190", "8925", "91988", "999995"]

ventas_2025_sin_bolsa = ventas_2025_sin_cliente_generico[~ventas_2025_sin_cliente_generico.producto_id.isin(excluir_producto_id)]

In [9]:
feats_g = features_clientes(ventas_2025_sin_bolsa, group_cols=("cliente_id_limpio",))
feats_g.describe()

C:\Users\Administrador\AppData\Local\Temp\2\ipykernel_25624\2727353546.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(list(group_cols) + ["rango_upt"])
C:\Users\Administrador\AppData\Local\Temp\2\ipykernel_25624\2727353546.py:165: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0.0)


,tickets_unicos,unidades_totales,ventas_totales,dias_con_compra,unidades_por_ticket,unidades_por_dia,tickets_total_upt,tickets_1_10,tickets_11_100,tickets_101_999,tickets_ge_1000,pct_tickets_1_10,pct_tickets_11_100,pct_tickets_101_999,pct_tickets_ge_1000,upt_p95,hhi_sku,top_share_sku,n_skus
count,421233.000000,421233.000000,4.212330e+05,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000,421233.000000
mean,11.507935,218.781700,1.876332e+06,10.111136,18.394016,20.182882,11.507935,5.661838,5.664475,0.177906,0.000328,0.533345,0.449048,0.017292,0.000055,31.277813,0.160087,0.227287,81.833617
std,26.614799,595.550892,4.918251e+06,21.062682,25.185320,28.279727,26.614799,17.391694,13.615861,1.330996,0.039881,0.388865,0.383116,0.096858,0.005941,42.310483,0.232230,0.233118,134.678163
min,1.000000,0.000000,0.000000e+00,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,1.000000,11.000000,1.232382e+05,1.000000,5.592857,6.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000,7.732500,0.025380,0.075973,8.000000
50%,3.000000,42.120000,4.191205e+05,3.000000,12.099500,13.108605,3.000000,1.000000,1.000000,0.000000,0.000000,0.500000,0.500000,0.000000,0.000000,20.300000,0.063857,0.138889,27.000000
75%,10.000000,169.495000,1.533886e+06,9.000000,23.320000,25.682500,10.000000,4.000000,4.000000,0.000000,0.000000,1.000000,0.800000,0.000000,0.000000,42.202500,0.183485,0.282125,91.000000
max,1602.000000,76687.380000,6.165853e+08,358.000000,3594.000000,4044.335000,1602.000000,1514.000000,458.000000,237.000000,14.000000,1.000000,1.000000,1.000000,1.000000,8018.000000,1.000000,1.000000,1929.000000


In [10]:
lab_g = etiquetar_mayoristas(feats_g, group_cols=("cliente_id_limpio",), umbral_score=0.75)
lab_g.head()

,cliente_id_limpio,tickets_unicos,unidades_totales,ventas_totales,dias_con_compra,unidades_por_ticket,unidades_por_dia,tickets_total_upt,tickets_1_10,tickets_11_100,...,pct_tickets_11_100,pct_tickets_101_999,pct_tickets_ge_1000,upt_p95,hhi_sku,top_share_sku,n_skus,mayorista_score,heuristicas_flag,mayorista_reglas
0,,1,3.000,15850.00,1,3.000000,3.000000,1,1,0,...,0.000000,0.0,0.0,3.0000,0.333333,0.333333,3,26.467080,True,True
1,0,469,1741.481,15733633.00,196,3.713179,8.885107,469,437,32,...,0.068230,0.0,0.0,13.6090,0.004514,0.028068,860,55.211011,True,True
2,00,467,2264.955,20493836.75,192,4.850011,11.796641,467,391,64,...,0.137045,0.0,0.0,19.0875,0.002447,0.013245,1222,56.859553,True,True
3,000,464,2362.952,21720658.05,203,5.092569,11.640158,464,390,59,...,0.127155,0.0,0.0,21.1045,0.003344,0.024546,1202,57.398067,True,True
4,0000,345,1386.170,12721300.50,160,4.017884,8.663563,345,308,30,...,0.086957,0.0,0.0,15.9790,0.003507,0.016592,819,55.566278,True,True


In [11]:
clientes_all = clasificar_mayoristas(ventas_2025_sin_bolsa, por_sucursal=True, incluir_global=True, umbral_score=75)

C:\Users\Administrador\AppData\Local\Temp\2\ipykernel_25624\2727353546.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(list(group_cols) + ["rango_upt"])
C:\Users\Administrador\AppData\Local\Temp\2\ipykernel_25624\2727353546.py:165: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0.0)
C:\Users\Administrador\AppData\Local\Temp\2\ipykernel_25624\2727353546.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future de

In [12]:
clientes_all.head(20)

,scope_pdv_id,cliente_id_limpio,mayorista_final,mayorista_reglas,mayorista_score,mayorista_score_ajuste_ml,iforest_flag,kmeans_flag,iforest_score,kmeans_cluster,...,tickets_ge_1000,tickets_total_upt,pct_tickets_1_10,pct_tickets_11_100,pct_tickets_101_999,pct_tickets_ge_1000,n_skus,hhi_sku,top_share_sku,heuristicas_flag
0,GLOBAL,,False,False,26.467080,26.467080,False,False,3.355491,1,...,0,1,1.000000,0.000000,0.0,0.0,3,0.333333,0.333333,False
1,GLOBAL,0,True,False,55.211011,65.211011,True,True,57.438615,0,...,0,469,0.931770,0.068230,0.0,0.0,860,0.004514,0.028068,False
2,GLOBAL,00,True,False,56.859553,66.859553,True,True,64.895968,0,...,0,467,0.837259,0.137045,0.0,0.0,1222,0.002447,0.013245,False
3,GLOBAL,000,True,False,57.398067,67.398067,True,True,65.935418,0,...,0,464,0.840517,0.127155,0.0,0.0,1202,0.003344,0.024546,False
4,GLOBAL,0000,False,False,55.566278,60.566278,True,False,49.659011,1,...,0,345,0.892754,0.086957,0.0,0.0,819,0.003507,0.016592,False
5,GLOBAL,00000,False,False,56.975332,61.975332,True,False,43.296757,1,...,0,183,0.825137,0.153005,0.0,0.0,629,0.004101,0.022716,False
6,GLOBAL,000000,False,False,53.787215,53.787215,False,False,29.957096,1,...,0,128,0.835938,0.125000,0.0,0.0,319,0.011904,0.051583,False
7,GLOBAL,0000000,False,False,50.916725,50.916725,False,False,19.038127,1,...,0,54,0.814815,0.148148,0.0,0.0,200,0.009320,0.040049,False
8,GLOBAL,00000000,False,False,43.711390,43.711390,False,False,4.946552,1,...,0,11,0.909091,0.090909,0.0,0.0,40,0.035183,0.099275,False
9,GLOBAL,000416890,False,False,53.705918,53.705918,False,False,9.064198,1,...,0,1,0.000000,1.000000,0.0,0.0,17,0.081797,0.130976,False


In [13]:
clientes_all.head()

,scope_pdv_id,cliente_id_limpio,mayorista_final,mayorista_reglas,mayorista_score,mayorista_score_ajuste_ml,iforest_flag,kmeans_flag,iforest_score,kmeans_cluster,...,tickets_ge_1000,tickets_total_upt,pct_tickets_1_10,pct_tickets_11_100,pct_tickets_101_999,pct_tickets_ge_1000,n_skus,hhi_sku,top_share_sku,heuristicas_flag
0,GLOBAL,,False,False,26.467080,26.467080,False,False,3.355491,1,...,0,1,1.000000,0.000000,0.0,0.0,3,0.333333,0.333333,False
1,GLOBAL,0,True,False,55.211011,65.211011,True,True,57.438615,0,...,0,469,0.931770,0.068230,0.0,0.0,860,0.004514,0.028068,False
2,GLOBAL,00,True,False,56.859553,66.859553,True,True,64.895968,0,...,0,467,0.837259,0.137045,0.0,0.0,1222,0.002447,0.013245,False
3,GLOBAL,000,True,False,57.398067,67.398067,True,True,65.935418,0,...,0,464,0.840517,0.127155,0.0,0.0,1202,0.003344,0.024546,False
4,GLOBAL,0000,False,False,55.566278,60.566278,True,False,49.659011,1,...,0,345,0.892754,0.086957,0.0,0.0,819,0.003507,0.016592,False


In [14]:
clientes_all.shape

(952012, 30)

In [15]:
# # Mostrar las 10 primeras filas de cada grupo (Global y cada sucursal)
# out = (
#     clientes_all
#     .groupby("scope_pdv_id", group_keys=True)
#     .head(10)
#     .sort_values(["scope_pdv_id", "mayorista_score_ajuste_ml"], ascending=[True, False])
# )

# out

In [16]:
# out.to_csv("resumen_clientes_mayoristas_2025.csv", index=False, encoding="utf-8")

In [17]:
# clientes_all.rename(columns=
#     {"wholesale_score":"mayorista_score",
#         "wholesale_score_adj": "mayorista_score_ajuste_ml"
#     }, inplace=True)

In [18]:
SQL_CLIENTES_LIMPIO = """
SELECT * FROM dbo.clientes_limpio
"""
with conexion_bd() as conn:
    clientes_limpio = pd.read_sql(SQL_CLIENTES_LIMPIO,conn)

clientes_limpio.rename(columns={"nombre":"Nombre"},inplace=True)
clientes_limpio.head()

C:\Users\Administrador\AppData\Local\Temp\2\ipykernel_25624\1197546919.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  clientes_limpio = pd.read_sql(SQL_CLIENTES_LIMPIO,conn)


,cliente_id_limpio,Nombre,ci_ruc
0,----,HUGO VEGA,----
1,MARIA,MARIA GRACIELA BERNAL,
2,********,ALDO GIMENEZ,********
3,***********,COMISION DE FESTEJOS PATRONALES DE LA C SL,***********
4,",4106091",JOSE LEIVA,",4106091-1"


In [19]:
clientes_mayoristas = pd.merge(clientes_all, clientes_limpio, on="cliente_id_limpio", how="left")

In [34]:
clientes_mayoristas['pdv_nombre'] = clientes_mayoristas['scope_pdv_id'].apply(add_branch_name)
clientes_mayoristas['pdv_nombre'].unique()

array(['General', 'Ahorrazo SL', 'Ahorrazo R1', 'Ahorrazo R2'],
      dtype=object)

In [35]:
clientes_mayoristas.shape

(952012, 33)

In [36]:
clientes_mayoristas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 952012 entries, 0 to 952011
Data columns (total 33 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   scope_pdv_id               952012 non-null  object 
 1   cliente_id_limpio          952012 non-null  object 
 2   mayorista_final            952012 non-null  bool   
 3   mayorista_reglas           952012 non-null  bool   
 4   mayorista_score            952012 non-null  float64
 5   mayorista_score_ajuste_ml  952012 non-null  float64
 6   iforest_flag               952012 non-null  bool   
 7   kmeans_flag                952012 non-null  bool   
 8   iforest_score              952012 non-null  float64
 9   kmeans_cluster             952012 non-null  int32  
 10  tickets_unicos             952012 non-null  int64  
 11  unidades_totales           952012 non-null  float64
 12  ventas_totales             952012 non-null  float64
 13  dias_con_compra            95

In [24]:
clientes_mayoristas.head()

,scope_pdv_id,cliente_id_limpio,mayorista_final,mayorista_reglas,mayorista_score,mayorista_score_ajuste_ml,iforest_flag,kmeans_flag,iforest_score,kmeans_cluster,...,pct_tickets_1_10,pct_tickets_11_100,pct_tickets_101_999,pct_tickets_ge_1000,n_skus,hhi_sku,top_share_sku,heuristicas_flag,Nombre,ci_ruc
0,GLOBAL,,False,False,26.467080,26.467080,False,False,3.355491,1,...,1.000000,0.000000,0.0,0.0,3,0.333333,0.333333,False,NaN,NaN
1,GLOBAL,0,True,False,55.211011,65.211011,True,True,57.438615,0,...,0.931770,0.068230,0.0,0.0,860,0.004514,0.028068,False,JOSE,0
2,GLOBAL,00,True,False,56.859553,66.859553,True,True,64.895968,0,...,0.837259,0.137045,0.0,0.0,1222,0.002447,0.013245,False,LUCAS,00
3,GLOBAL,000,True,False,57.398067,67.398067,True,True,65.935418,0,...,0.840517,0.127155,0.0,0.0,1202,0.003344,0.024546,False,LUIS,000
4,GLOBAL,0000,False,False,55.566278,60.566278,True,False,49.659011,1,...,0.892754,0.086957,0.0,0.0,819,0.003507,0.016592,False,SIN NOMBRE,0000


In [42]:
def engine_sqlserver():
    drivers = pyodbc.drivers()
    preferidos = ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server"]
    driver = next((d for d in preferidos if d in drivers), None)
    if not driver:
        raise RuntimeError(f"No se encontró driver ODBC SQL Server. Detectados: {drivers}")

    odbc_str = (
        f"DRIVER={{{driver}}};"
        f"SERVER=192.168.5.231;"
        f"DATABASE=DATO_SOLUTIONS;"
        f"UID=sa;"
        f"PWD=Server_123456;"
        f"Encrypt=yes;"
        f"TrustServerCertificate=yes;"
    )

    params = quote_plus(odbc_str)
    return create_engine(
        f"mssql+pyodbc:///?odbc_connect={params}",
        fast_executemany=True
    )

In [44]:
table_name = 'clientes_mayoristas'
engine = engine_sqlserver()
try:
    clientes_mayoristas.to_sql(
        table_name,
        schema="dbo",
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=50000
    )
    print(f"Dataframe successfully written to table {table_name}")
except Exception as e:
    print(f"An error ocurred: {e}")

Dataframe successfully written to table clientes_mayoristas


In [21]:
clientes_mayoristas.to_csv("clientes_mayoristas_2025.csv", index=False, encoding="utf-8")